<a href="https://colab.research.google.com/github/Nirmitdagli/llm-kernels/blob/main/llm_kernels_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Kernels on a free Colab GPU

**First:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all.

1. Triton softmax / RMSNorm / SwiGLU: forward + backward tests, then benchmarks
2. CUDA GEMM: naive → tiled → register-blocked → float4, vs cuBLAS
3. JAX comparison

Copy every results table into README.md, then push the folder to GitHub.

In [1]:
!nvidia-smi
import torch, triton
print(torch.__version__, triton.__version__, torch.cuda.get_device_name())

Fri Oct  9 05:33:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Triton kernels (forward + backward)
Read every comment in the kernel file before running it.

In [2]:
%%writefile triton_kernels.py
"""
Fused LLM kernels in Triton: softmax, RMSNorm, SwiGLU. Forward AND backward.

Each kernel replaces several PyTorch ops (several trips to GPU memory) with ONE
read and ONE write. These ops are memory-bound, so fewer trips = faster.

Triton model in one line: you write what ONE program (block) does on a whole
tile of data. Triton maps it to threads, coalesced loads and shared memory.

PyTorch integration: each op is a torch.autograd.Function, so it can be used
inside a model and trained (loss.backward() calls our backward kernels).
"""
import os
import torch
import triton
import triton.language as tl

_INTERPRET = os.environ.get("TRITON_INTERPRET") == "1"   # CPU interpreter for testing without a GPU


def _row_warps(block):
    """More warps for wider rows so each thread handles fewer elements."""
    return 4 if block <= 2048 else (8 if block <= 8192 else 16)


# =============================================================================
# 1. SOFTMAX over the last dimension
#    forward : y  = exp(x - max) / sum(exp(x - max))
#    backward: dx = y * (dy - sum(dy * y))          (per row)
#    One program per row. The whole row fits in one block (BLOCK >= n_cols).
# =============================================================================
@triton.jit
def softmax_fwd_kernel(x_ptr, y_ptr, stride, n_cols, BLOCK: tl.constexpr):
    row = tl.program_id(0)                       # which row this program owns
    cols = tl.arange(0, BLOCK)                   # column indices 0..BLOCK-1
    mask = cols < n_cols                         # BLOCK is a power of 2; the row may be shorter

    # Padding = -inf so it never wins the max, and exp(-inf) = 0 adds nothing to the sum.
    x = tl.load(x_ptr + row * stride + cols, mask=mask, other=-float("inf")).to(tl.float32)
    x = x - tl.max(x, axis=0)                    # numerical stability: largest value becomes 0
    num = tl.exp(x)
    y = num / tl.sum(num, axis=0)
    tl.store(y_ptr + row * stride + cols, y.to(y_ptr.dtype.element_ty), mask=mask)


@triton.jit
def softmax_bwd_kernel(y_ptr, dy_ptr, dx_ptr, stride, n_cols, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    mask = cols < n_cols
    y = tl.load(y_ptr + row * stride + cols, mask=mask, other=0.0).to(tl.float32)
    dy = tl.load(dy_ptr + row * stride + cols, mask=mask, other=0.0).to(tl.float32)
    dot = tl.sum(dy * y, axis=0)                 # one scalar per row
    dx = y * (dy - dot)
    tl.store(dx_ptr + row * stride + cols, dx.to(dx_ptr.dtype.element_ty), mask=mask)


class TritonSoftmax(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        shape = x.shape
        x2 = x.reshape(-1, shape[-1]).contiguous()
        rows, cols = x2.shape
        y = torch.empty_like(x2)
        BLOCK = triton.next_power_of_2(cols)
        softmax_fwd_kernel[(rows,)](x2, y, x2.stride(0), cols, BLOCK=BLOCK, num_warps=_row_warps(BLOCK))
        ctx.save_for_backward(y)                 # backward only needs the output y
        ctx.shape = shape
        return y.reshape(shape)

    @staticmethod
    def backward(ctx, dy):
        (y,) = ctx.saved_tensors
        dy2 = dy.reshape(y.shape).contiguous()
        rows, cols = y.shape
        dx = torch.empty_like(y)
        BLOCK = triton.next_power_of_2(cols)
        softmax_bwd_kernel[(rows,)](y, dy2, dx, y.stride(0), cols, BLOCK=BLOCK, num_warps=_row_warps(BLOCK))
        return dx.reshape(ctx.shape)


def softmax(x):
    return TritonSoftmax.apply(x)


# =============================================================================
# 2. RMSNorm (Llama, Mistral, DeepSeek)
#    forward : rstd = 1 / sqrt(mean(x^2) + eps);  xhat = x * rstd;  y = xhat * w
#    backward: dx = rstd * (dy*w - xhat * mean(dy*w*xhat))
#              dw = sum over rows of (dy * xhat)
#    One program per row (one token's hidden vector).
# =============================================================================
@triton.jit
def rmsnorm_fwd_kernel(x_ptr, w_ptr, y_ptr, rstd_ptr, stride, n_cols, eps, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    mask = cols < n_cols

    x = tl.load(x_ptr + row * stride + cols, mask=mask, other=0.0).to(tl.float32)
    w = tl.load(w_ptr + cols, mask=mask, other=0.0).to(tl.float32)

    mean_sq = tl.sum(x * x, axis=0) / n_cols     # padding is 0, so it does not change the sum
    rstd = 1.0 / tl.sqrt(mean_sq + eps)          # reciprocal standard deviation
    tl.store(rstd_ptr + row, rstd)               # saved for the backward pass (1 float per row)
    y = x * rstd * w
    tl.store(y_ptr + row * stride + cols, y.to(y_ptr.dtype.element_ty), mask=mask)


@triton.jit
def rmsnorm_bwd_kernel(x_ptr, w_ptr, dy_ptr, rstd_ptr, dx_ptr, dw_part_ptr, stride, n_cols,
                       BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    mask = cols < n_cols

    x = tl.load(x_ptr + row * stride + cols, mask=mask, other=0.0).to(tl.float32)
    w = tl.load(w_ptr + cols, mask=mask, other=0.0).to(tl.float32)
    dy = tl.load(dy_ptr + row * stride + cols, mask=mask, other=0.0).to(tl.float32)
    rstd = tl.load(rstd_ptr + row)

    xhat = x * rstd
    dyw = dy * w
    c = tl.sum(dyw * xhat, axis=0) / n_cols
    dx = (dyw - xhat * c) * rstd
    tl.store(dx_ptr + row * stride + cols, dx.to(dx_ptr.dtype.element_ty), mask=mask)

    # dw needs a sum over ALL rows. Each program writes its row's share (fp32);
    # the host sums the [rows, cols] partials with one torch.sum. Simple and deterministic.
    tl.store(dw_part_ptr + row * n_cols + cols, dy * xhat, mask=mask)


class TritonRMSNormFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, w, eps):
        shape = x.shape
        x2 = x.reshape(-1, shape[-1]).contiguous()   # [batch, seq, hidden] -> [tokens, hidden]
        rows, cols = x2.shape
        y = torch.empty_like(x2)
        rstd = torch.empty(rows, device=x.device, dtype=torch.float32)
        BLOCK = triton.next_power_of_2(cols)
        rmsnorm_fwd_kernel[(rows,)](x2, w, y, rstd, x2.stride(0), cols, eps,
                                    BLOCK=BLOCK, num_warps=_row_warps(BLOCK))
        ctx.save_for_backward(x2, w, rstd)
        ctx.shape = shape
        return y.reshape(shape)

    @staticmethod
    def backward(ctx, dy):
        x2, w, rstd = ctx.saved_tensors
        rows, cols = x2.shape
        dy2 = dy.reshape(rows, cols).contiguous()
        dx = torch.empty_like(x2)
        dw_part = torch.empty(rows, cols, device=x2.device, dtype=torch.float32)
        BLOCK = triton.next_power_of_2(cols)
        rmsnorm_bwd_kernel[(rows,)](x2, w, dy2, rstd, dx, dw_part, x2.stride(0), cols,
                                    BLOCK=BLOCK, num_warps=_row_warps(BLOCK))
        dw = dw_part.sum(0).to(w.dtype)
        return dx.reshape(ctx.shape), dw, None    # no gradient for eps


def rmsnorm(x, weight, eps=1e-6):
    return TritonRMSNormFn.apply(x, weight, eps)


class TritonRMSNorm(torch.nn.Module):
    """Drop-in replacement for the RMSNorm layer in Llama-style models."""
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = torch.nn.Parameter(torch.ones(hidden_size))
        self.eps = eps

    def forward(self, x):
        return rmsnorm(x, self.weight, self.eps)


# =============================================================================
# 3. SwiGLU (the MLP activation in Llama): y = silu(gate) * up
#    silu(g) = g * sigmoid(g),  silu'(g) = sig * (1 + g * (1 - sig))
#    backward: d_gate = dy * up * silu'(gate);  d_up = dy * silu(gate)
#    Pure elementwise: treat tensors as flat 1D arrays, tile them, AUTOTUNE the block size.
# =============================================================================
@triton.jit
def _swiglu_fwd_kernel(g_ptr, u_ptr, y_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    g = tl.load(g_ptr + offs, mask=mask).to(tl.float32)
    u = tl.load(u_ptr + offs, mask=mask).to(tl.float32)
    y = g * tl.sigmoid(g) * u                    # fused: 2 reads + 1 write instead of 3 kernels
    tl.store(y_ptr + offs, y.to(y_ptr.dtype.element_ty), mask=mask)


@triton.jit
def _swiglu_bwd_kernel(g_ptr, u_ptr, dy_ptr, dg_ptr, du_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    g = tl.load(g_ptr + offs, mask=mask).to(tl.float32)
    u = tl.load(u_ptr + offs, mask=mask).to(tl.float32)
    dy = tl.load(dy_ptr + offs, mask=mask).to(tl.float32)
    sig = tl.sigmoid(g)
    silu = g * sig
    dsilu = sig * (1.0 + g * (1.0 - sig))
    tl.store(dg_ptr + offs, (dy * u * dsilu).to(dg_ptr.dtype.element_ty), mask=mask)
    tl.store(du_ptr + offs, (dy * silu).to(du_ptr.dtype.element_ty), mask=mask)


# Autotune tries each config the first time a size is seen and caches the fastest.
# It needs a real GPU, so the CPU interpreter uses a fixed block size instead.
_EW_CONFIGS = [
    triton.Config({"BLOCK": 512}, num_warps=4),
    triton.Config({"BLOCK": 1024}, num_warps=4),
    triton.Config({"BLOCK": 2048}, num_warps=8),
    triton.Config({"BLOCK": 4096}, num_warps=8),
]
if _INTERPRET:
    swiglu_fwd_kernel, swiglu_bwd_kernel = _swiglu_fwd_kernel, _swiglu_bwd_kernel
else:
    swiglu_fwd_kernel = triton.autotune(configs=_EW_CONFIGS, key=["n"])(_swiglu_fwd_kernel)
    swiglu_bwd_kernel = triton.autotune(configs=_EW_CONFIGS, key=["n"])(_swiglu_bwd_kernel)


def _launch_ew(kernel, n, *args):
    grid = lambda meta: (triton.cdiv(n, meta["BLOCK"]),)
    if _INTERPRET:
        kernel[grid](*args, n, BLOCK=1024)
    else:
        kernel[grid](*args, n)


class TritonSwiGLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, gate, up):
        assert gate.shape == up.shape
        gate, up = gate.contiguous(), up.contiguous()
        y = torch.empty_like(gate)
        _launch_ew(swiglu_fwd_kernel, gate.numel(), gate, up, y)
        ctx.save_for_backward(gate, up)
        return y

    @staticmethod
    def backward(ctx, dy):
        gate, up = ctx.saved_tensors
        dy = dy.contiguous()
        dg, du = torch.empty_like(gate), torch.empty_like(up)
        _launch_ew(swiglu_bwd_kernel, gate.numel(), gate, up, dy, dg, du)
        return dg, du


def swiglu(gate, up):
    return TritonSwiGLU.apply(gate, up)


# =============================================================================
# PyTorch references (what we check against and benchmark against)
# =============================================================================
def softmax_ref(x):
    return torch.softmax(x.float(), dim=-1).to(x.dtype)


def rmsnorm_ref(x, w, eps=1e-6):
    xf = x.float()
    return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps) * w.float()).to(x.dtype)


def swiglu_ref(g, u):
    return (torch.nn.functional.silu(g.float()) * u.float()).to(g.dtype)


Writing triton_kernels.py


In [3]:
%%writefile test_and_bench.py
"""
Correctness tests (forward + backward) and benchmarks for the Triton kernels.

On a GPU (Colab T4 works):  python test_and_bench.py
Without a GPU (tests only): TRITON_INTERPRET=1 python test_and_bench.py --test-only
"""
import os
import sys
import torch
import triton

from triton_kernels import (softmax, rmsnorm, swiglu,
                            softmax_ref, rmsnorm_ref, swiglu_ref)

INTERPRET = os.environ.get("TRITON_INTERPRET") == "1"
DEVICE = "cpu" if INTERPRET else "cuda"


def check(name, ours, ref, atol, rtol):
    torch.testing.assert_close(ours, ref, atol=atol, rtol=rtol, msg=lambda m: f"{name}: {m}")


def test():
    """Compare outputs AND gradients with PyTorch autograd, in fp32 and (on GPU) fp16."""
    torch.manual_seed(0)
    dtypes = [torch.float32] if INTERPRET else [torch.float32, torch.float16]
    for dtype in dtypes:
        tol = 1e-4 if dtype == torch.float32 else 2e-2
        for rows, cols in [(4, 7), (16, 128), (8, 1000)]:      # odd sizes exercise the masks
            # --- softmax ---
            x = torch.randn(rows, cols, device=DEVICE, dtype=dtype, requires_grad=True)
            x_ref = x.detach().clone().requires_grad_(True)
            dy = torch.randn(rows, cols, device=DEVICE, dtype=dtype)
            y, y_ref = softmax(x), softmax_ref(x_ref)
            check("softmax fwd", y, y_ref, tol, tol)
            y.backward(dy); y_ref.backward(dy)
            check("softmax bwd dx", x.grad, x_ref.grad, tol, tol)

            # --- RMSNorm (3D input, like [batch, seq, hidden]) ---
            x = torch.randn(2, rows, cols, device=DEVICE, dtype=dtype, requires_grad=True)
            w = torch.randn(cols, device=DEVICE, dtype=dtype, requires_grad=True)
            x_ref = x.detach().clone().requires_grad_(True)
            w_ref = w.detach().clone().requires_grad_(True)
            dy = torch.randn(2, rows, cols, device=DEVICE, dtype=dtype)
            y, y_ref = rmsnorm(x, w), rmsnorm_ref(x_ref, w_ref)
            check("rmsnorm fwd", y, y_ref, tol, tol)
            y.backward(dy); y_ref.backward(dy)
            check("rmsnorm bwd dx", x.grad, x_ref.grad, 5 * tol, 5 * tol)
            check("rmsnorm bwd dw", w.grad, w_ref.grad, 5 * tol, 5 * tol)

            # --- SwiGLU ---
            g = torch.randn(rows, cols, device=DEVICE, dtype=dtype, requires_grad=True)
            u = torch.randn(rows, cols, device=DEVICE, dtype=dtype, requires_grad=True)
            g_ref = g.detach().clone().requires_grad_(True)
            u_ref = u.detach().clone().requires_grad_(True)
            dy = torch.randn(rows, cols, device=DEVICE, dtype=dtype)
            y, y_ref = swiglu(g, u), swiglu_ref(g_ref, u_ref)
            check("swiglu fwd", y, y_ref, tol, tol)
            y.backward(dy); y_ref.backward(dy)
            check("swiglu bwd d_gate", g.grad, g_ref.grad, tol, tol)
            check("swiglu bwd d_up", u.grad, u_ref.grad, tol, tol)
    print("All forward and backward correctness tests passed.")


def gbps(nbytes, ms):
    return nbytes / (ms * 1e-3) / 1e9


def bench():
    """Memory-bound kernels: report GB/s. Triton vs PyTorch eager vs torch.compile, fp16."""
    dtype, rows = torch.float16, 4096                       # 4096 tokens per batch
    c_soft, c_rms, c_swi = (torch.compile(f) for f in (softmax_ref, rmsnorm_ref, swiglu_ref))
    print(f"\nGPU: {torch.cuda.get_device_name()}  |  {rows} rows, fp16, forward pass")
    print("| Kernel | Hidden size | Triton ms | Eager ms | torch.compile ms | Triton GB/s | Eager GB/s | Speedup vs eager |")
    print("| --- | --- | --- | --- | --- | --- | --- | --- |")
    for cols in [1024, 2048, 4096, 8192]:                   # typical hidden sizes
        x = torch.randn(rows, cols, device="cuda", dtype=dtype)
        w = torch.randn(cols, device="cuda", dtype=dtype)
        g = torch.randn(rows, cols, device="cuda", dtype=dtype)
        u = torch.randn(rows, cols, device="cuda", dtype=dtype)
        e = x.element_size()
        cases = [
            ("softmax", 2 * x.numel() * e, lambda: softmax(x), lambda: softmax_ref(x), lambda: c_soft(x)),
            ("rmsnorm", 2 * x.numel() * e, lambda: rmsnorm(x, w), lambda: rmsnorm_ref(x, w), lambda: c_rms(x, w)),
            ("swiglu", 3 * x.numel() * e, lambda: swiglu(g, u), lambda: swiglu_ref(g, u), lambda: c_swi(g, u)),
        ]
        with torch.no_grad():
            for name, nbytes, f_t, f_e, f_c in cases:
                t, e_, c = (triton.testing.do_bench(f) for f in (f_t, f_e, f_c))
                print(f"| {name} | {cols} | {t:.3f} | {e_:.3f} | {c:.3f} | {gbps(nbytes, t):.0f} | {gbps(nbytes, e_):.0f} | {e_ / t:.2f}x |")

    # Forward + backward (training step) for RMSNorm, the most common use.
    print("\n| RMSNorm fwd+bwd | Hidden size | Triton ms | Eager ms | Speedup |")
    print("| --- | --- | --- | --- | --- |")
    for cols in [1024, 4096, 8192]:
        x = torch.randn(rows, cols, device="cuda", dtype=dtype, requires_grad=True)
        w = torch.randn(cols, device="cuda", dtype=dtype, requires_grad=True)
        dy = torch.randn(rows, cols, device="cuda", dtype=dtype)
        t = triton.testing.do_bench(lambda: rmsnorm(x, w).backward(dy), grad_to_none=[x, w])
        e_ = triton.testing.do_bench(lambda: rmsnorm_ref(x, w).backward(dy), grad_to_none=[x, w])
        print(f"| rmsnorm | {cols} | {t:.3f} | {e_:.3f} | {e_ / t:.2f}x |")


if __name__ == "__main__":
    test()
    if "--test-only" not in sys.argv and not INTERPRET:
        bench()


Writing test_and_bench.py


In [4]:
!python test_and_bench.py

All forward and backward correctness tests passed.

GPU: Tesla T4  |  4096 rows, fp16, forward pass
| Kernel | Hidden size | Triton ms | Eager ms | torch.compile ms | Triton GB/s | Eager GB/s | Speedup vs eager |
| --- | --- | --- | --- | --- | --- | --- | --- |
| softmax | 1024 | 0.073 | 0.420 | 0.079 | 229 | 40 | 5.75x |
| rmsnorm | 1024 | 0.075 | 0.762 | 0.076 | 223 | 22 | 10.12x |
| swiglu | 1024 | 0.104 | 0.702 | 0.104 | 243 | 36 | 6.78x |
/usr/local/lib/python3.13/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
| softmax | 2048 | 0.147 | 0.758 | 0.148 | 228 | 44 | 5.14x |
| rmsnorm | 2048 | 0.149 | 1.445 | 0.146 | 225 | 23 | 9.71x |
| swiglu | 2048 | 0.204 | 1.395 | 0.201 | 246 | 36 | 6.82x |
| softmax | 4096 | 0.292 | 1.434 | 0.293 | 230 | 47 | 4.91x |
| rmsno

## 2. GEMM in CUDA C++
`sm_75` is the T4. An L4 is `sm_89`, an A100 is `sm_80`.

In [5]:
%%writefile gemm.cu
// Tiled GEMM in CUDA C++, optimized step by step and measured against cuBLAS:
//   1. naive  2. shared-memory tiling  3. 2D register blocking  4. float4 vectorized loads
// C = A x B, all N x N, row-major, fp32.
//
// Build and run (Colab T4):
//   nvcc -O3 -arch=sm_75 gemm.cu -lcublas -o gemm && ./gemm
// (sm_75 = T4. Use sm_80 for A100, sm_89 for L4.)

#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>
#include <cuda_runtime.h>
#include <cublas_v2.h>

#define CHECK(x) do { cudaError_t e = (x); if (e != cudaSuccess) { \
    printf("CUDA error %s at %s:%d\n", cudaGetErrorString(e), __FILE__, __LINE__); exit(1); } } while (0)
#define CHECK_CUBLAS(x) do { cublasStatus_t st = (x); if (st != CUBLAS_STATUS_SUCCESS) { \
    printf("cuBLAS error %d at %s:%d\n", (int)st, __FILE__, __LINE__); exit(1); } } while (0)

// ---------------------------------------------------------------------------
// Kernel 1: naive. One thread computes one C[row][col].
// Every multiply reads A and B straight from global memory (HBM):
// for N = 4096 that is 2 x 4096 global loads per output. Very low reuse.
// ---------------------------------------------------------------------------
__global__ void gemm_naive(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;   // x walks columns -> B and C loads are coalesced
    if (row < N && col < N) {
        float acc = 0.0f;
        for (int k = 0; k < N; ++k)
            acc += A[row * N + k] * B[k * N + col];
        C[row * N + col] = acc;
    }
}

// ---------------------------------------------------------------------------
// Kernel 2: shared-memory tiling.
// A block of TILE x TILE threads computes a TILE x TILE patch of C.
// It walks along K in steps of TILE: each step loads one tile of A and one
// tile of B into shared memory (one element per thread), syncs, then every
// thread reuses those 2*TILE values from fast on-chip memory.
// Global loads drop by a factor of TILE.
// ---------------------------------------------------------------------------
template <int TILE>
__global__ void gemm_tiled(const float* A, const float* B, float* C, int N) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE + ty;
    int col = blockIdx.x * TILE + tx;
    float acc = 0.0f;

    for (int t = 0; t < (N + TILE - 1) / TILE; ++t) {
        // Cooperative load: each thread brings in one element of each tile.
        int a_col = t * TILE + tx;
        int b_row = t * TILE + ty;
        As[ty][tx] = (row < N && a_col < N) ? A[row * N + a_col] : 0.0f;
        Bs[ty][tx] = (b_row < N && col < N) ? B[b_row * N + col] : 0.0f;
        __syncthreads();                     // 1st sync: tiles fully loaded before anyone reads

        #pragma unroll
        for (int k = 0; k < TILE; ++k)
            acc += As[ty][k] * Bs[k][tx];    // all reads hit shared memory
        __syncthreads();                     // 2nd sync: nobody overwrites a tile still being read
    }
    if (row < N && col < N) C[row * N + col] = acc;
}

// ---------------------------------------------------------------------------
// Kernel 3: 2D register blocking (block tile 64x64, thread tile 4x4).
// Tiling alone still does 2 shared-memory loads per multiply-add.
// Now each thread computes a 4x4 patch of C, so per k step it loads
// 4 values of A + 4 of B into REGISTERS and does 16 multiply-adds:
// 8 loads per 16 FMAs instead of 2 per 1. 256 threads per block.
// Requires N to be a multiple of 64 (true for all sizes we benchmark).
// ---------------------------------------------------------------------------
constexpr int BM = 64, BN = 64, TM = 4, TN = 4;          // block tile, thread tile
constexpr int THREADS = (BM / TM) * (BN / TN);           // 16 x 16 = 256

template <int BK>
__global__ void gemm_regblock(const float* A, const float* B, float* C, int N) {
    __shared__ float As[BM][BK];
    __shared__ float Bs[BK][BN];

    const int tid = threadIdx.x;
    const int tRow = tid / (BN / TN);                     // 0..15: which 4-row strip of the tile
    const int tCol = tid % (BN / TN);                     // 0..15: which 4-col strip of the tile
    const float* Ablk = A + blockIdx.y * BM * N;          // top-left of this block's rows of A
    const float* Bblk = B + blockIdx.x * BN;              // top-left of this block's cols of B
    float* Cblk = C + blockIdx.y * BM * N + blockIdx.x * BN;

    float acc[TM][TN] = {};
    float regA[TM], regB[TN];

    for (int k0 = 0; k0 < N; k0 += BK) {
        // Cooperative load: BM*BK elements of A and BK*BN of B, spread over 256 threads.
        for (int i = tid; i < BM * BK; i += THREADS)
            As[i / BK][i % BK] = Ablk[(i / BK) * N + k0 + i % BK];
        for (int i = tid; i < BK * BN; i += THREADS)
            Bs[i / BN][i % BN] = Bblk[(k0 + i / BN) * N + i % BN];   // consecutive threads -> consecutive columns: coalesced
        __syncthreads();

        #pragma unroll
        for (int k = 0; k < BK; ++k) {
            #pragma unroll
            for (int m = 0; m < TM; ++m) regA[m] = As[tRow * TM + m][k];
            #pragma unroll
            for (int n = 0; n < TN; ++n) regB[n] = Bs[k][tCol * TN + n];
            #pragma unroll
            for (int m = 0; m < TM; ++m)
                #pragma unroll
                for (int n = 0; n < TN; ++n)
                    acc[m][n] += regA[m] * regB[n];       // 16 FMAs from 8 register values
        }
        __syncthreads();
    }
    for (int m = 0; m < TM; ++m)
        for (int n = 0; n < TN; ++n)
            Cblk[(tRow * TM + m) * N + tCol * TN + n] = acc[m][n];
}

// ---------------------------------------------------------------------------
// Kernel 4: register blocking + float4 VECTORIZED loads and stores.
// Same 64x64 block tile and 4x4 thread tile, BK = 16, 256 threads.
// - Global loads: each thread reads ONE float4 (16 bytes) of A and ONE of B per
//   k step: 4x fewer load instructions than scalar loads.
// - A is stored TRANSPOSED in shared memory (AsT[k][m]) so a thread's 4 A values
//   for a given k are contiguous and can be read as one float4.
// - C is written back with float4 stores.
// Requires N % 64 == 0 (also guarantees 16-byte alignment of every float4).
// ---------------------------------------------------------------------------
constexpr int VBK = 16;

__global__ void gemm_vec4(const float* A, const float* B, float* C, int N) {
    __shared__ __align__(16) float AsT[VBK][BM];          // transposed A tile
    __shared__ __align__(16) float Bs[VBK][BN];

    const int tid = threadIdx.x;
    const int tRow = tid / (BN / TN);
    const int tCol = tid % (BN / TN);
    const float* Ablk = A + blockIdx.y * BM * N;
    const float* Bblk = B + blockIdx.x * BN;
    float* Cblk = C + blockIdx.y * BM * N + blockIdx.x * BN;

    // Which float4 this thread loads: A tile is 64 rows x 16 cols = 64 x 4 float4s,
    // B tile is 16 rows x 64 cols = 16 x 16 float4s. Both are exactly 256 float4s.
    const int aRow = tid / (VBK / 4), aCol = (tid % (VBK / 4)) * 4;
    const int bRow = tid / (BN / 4),  bCol = (tid % (BN / 4)) * 4;

    float acc[TM][TN] = {};

    for (int k0 = 0; k0 < N; k0 += VBK) {
        float4 a = *reinterpret_cast<const float4*>(&Ablk[aRow * N + k0 + aCol]);
        AsT[aCol + 0][aRow] = a.x;                         // scatter into the transposed tile
        AsT[aCol + 1][aRow] = a.y;
        AsT[aCol + 2][aRow] = a.z;
        AsT[aCol + 3][aRow] = a.w;
        *reinterpret_cast<float4*>(&Bs[bRow][bCol]) =
            *reinterpret_cast<const float4*>(&Bblk[(k0 + bRow) * N + bCol]);
        __syncthreads();

        #pragma unroll
        for (int k = 0; k < VBK; ++k) {
            float4 ra = *reinterpret_cast<const float4*>(&AsT[k][tRow * TM]);  // 4 A values, 1 load
            float4 rb = *reinterpret_cast<const float4*>(&Bs[k][tCol * TN]);   // 4 B values, 1 load
            float av[4] = {ra.x, ra.y, ra.z, ra.w};
            float bv[4] = {rb.x, rb.y, rb.z, rb.w};
            #pragma unroll
            for (int m = 0; m < TM; ++m)
                #pragma unroll
                for (int n = 0; n < TN; ++n)
                    acc[m][n] += av[m] * bv[n];
        }
        __syncthreads();
    }
    for (int m = 0; m < TM; ++m)
        *reinterpret_cast<float4*>(&Cblk[(tRow * TM + m) * N + tCol * TN]) =
            make_float4(acc[m][0], acc[m][1], acc[m][2], acc[m][3]);
}

// ---------------------------------------------------------------------------
// Timing helpers
// ---------------------------------------------------------------------------
template <typename F>
float time_ms(F launch, int iters = 10) {
    launch();                                // warm-up (first launch pays setup costs)
    CHECK(cudaGetLastError());               // a bad launch config fails silently otherwise
    CHECK(cudaDeviceSynchronize());
    cudaEvent_t s, e;
    cudaEventCreate(&s); cudaEventCreate(&e);
    cudaEventRecord(s);
    for (int i = 0; i < iters; ++i) launch();
    cudaEventRecord(e);
    CHECK(cudaGetLastError());
    cudaEventSynchronize(e);
    float ms = 0; cudaEventElapsedTime(&ms, s, e);
    cudaEventDestroy(s); cudaEventDestroy(e);
    return ms / iters;
}

float max_abs_diff(const std::vector<float>& a, const std::vector<float>& b) {
    float m = 0;
    for (size_t i = 0; i < a.size(); ++i) m = fmaxf(m, fabsf(a[i] - b[i]));
    return m;
}

int main() {
    cublasHandle_t handle;
    cublasCreate(&handle);
    // "Effective GB/s" = minimum bytes a GEMM must move (read A and B, write C) / time.
    // It shows how far each kernel is from being limited by memory bandwidth.
    printf("| N | Kernel | ms | GFLOP/s | Effective GB/s | %% of cuBLAS | max error vs cuBLAS |\n");
    printf("| --- | --- | --- | --- | --- | --- | --- |\n");

    for (int N : {512, 1024, 2048, 4096}) {
        size_t bytes = (size_t)N * N * sizeof(float);
        std::vector<float> hA(N * N), hB(N * N), ref(N * N), out(N * N);
        for (auto& v : hA) v = (rand() % 100) / 100.0f - 0.5f;
        for (auto& v : hB) v = (rand() % 100) / 100.0f - 0.5f;

        float *A, *B, *C;
        CHECK(cudaMalloc(&A, bytes)); CHECK(cudaMalloc(&B, bytes)); CHECK(cudaMalloc(&C, bytes));
        CHECK(cudaMemcpy(A, hA.data(), bytes, cudaMemcpyHostToDevice));
        CHECK(cudaMemcpy(B, hB.data(), bytes, cudaMemcpyHostToDevice));
        double flops = 2.0 * N * N * N;      // one multiply + one add per inner step

        // cuBLAS is column-major. Row-major C = A*B equals column-major C^T = B^T * A^T,
        // so we pass B first, then A, and get row-major C back.
        float alpha = 1.0f, beta = 0.0f;
        auto run_cublas = [&] {
            CHECK_CUBLAS(cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, N, N, &alpha, B, N, A, N, &beta, C, N));
        };
        float ms_cublas = time_ms(run_cublas);
        CHECK(cudaMemcpy(ref.data(), C, bytes, cudaMemcpyDeviceToHost));

        dim3 blk16(16, 16), grid16((N + 15) / 16, (N + 15) / 16);
        dim3 blk32(32, 32), grid32((N + 31) / 32, (N + 31) / 32);

        struct Case { const char* name; float ms; float err; };
        std::vector<Case> cases;

        CHECK(cudaMemset(C, 0, bytes));                  // stale output must not pass the check
        float ms = time_ms([&] { gemm_naive<<<grid16, blk16>>>(A, B, C, N); });
        CHECK(cudaMemcpy(out.data(), C, bytes, cudaMemcpyDeviceToHost));
        cases.push_back({"naive", ms, max_abs_diff(out, ref)});

        CHECK(cudaMemset(C, 0, bytes));                  // stale output must not pass the check
        ms = time_ms([&] { gemm_tiled<16><<<grid16, blk16>>>(A, B, C, N); });
        CHECK(cudaMemcpy(out.data(), C, bytes, cudaMemcpyDeviceToHost));
        cases.push_back({"tiled 16", ms, max_abs_diff(out, ref)});

        CHECK(cudaMemset(C, 0, bytes));                  // stale output must not pass the check
        ms = time_ms([&] { gemm_tiled<32><<<grid32, blk32>>>(A, B, C, N); });
        CHECK(cudaMemcpy(out.data(), C, bytes, cudaMemcpyDeviceToHost));
        cases.push_back({"tiled 32", ms, max_abs_diff(out, ref)});

        dim3 blkRB(THREADS), gridRB(N / BN, N / BM);    // N is a multiple of 64 for every size we run
        CHECK(cudaMemset(C, 0, bytes));                  // stale output must not pass the check
        ms = time_ms([&] { gemm_regblock<8><<<gridRB, blkRB>>>(A, B, C, N); });
        CHECK(cudaMemcpy(out.data(), C, bytes, cudaMemcpyDeviceToHost));
        cases.push_back({"regblock 4x4", ms, max_abs_diff(out, ref)});

        CHECK(cudaMemset(C, 0, bytes));                  // stale output must not pass the check
        ms = time_ms([&] { gemm_vec4<<<gridRB, blkRB>>>(A, B, C, N); });
        CHECK(cudaMemcpy(out.data(), C, bytes, cudaMemcpyDeviceToHost));
        cases.push_back({"vec4 + regblock", ms, max_abs_diff(out, ref)});

        cases.push_back({"cuBLAS", ms_cublas, 0.0f});

        for (auto& c : cases)
            printf("| %d | %s | %.3f | %.0f | %.0f | %.1f%% | %.2e |\n", N, c.name, c.ms,
                   flops / (c.ms * 1e-3) / 1e9, 3.0 * bytes / (c.ms * 1e-3) / 1e9,
                   100.0 * ms_cublas / c.ms, c.err);

        cudaFree(A); cudaFree(B); cudaFree(C);
    }
    cublasDestroy(handle);
    return 0;
}


Writing gemm.cu


In [6]:
!nvcc -O3 -arch=sm_75 gemm.cu -lcublas -o gemm && ./gemm

| N | Kernel | ms | GFLOP/s | Effective GB/s | % of cuBLAS | max error vs cuBLAS |
| --- | --- | --- | --- | --- | --- | --- |
| 512 | naive | 0.904 | 297 | 3 | 11.8% | 0.00e+00 |
| 512 | tiled 16 | 0.583 | 460 | 5 | 18.2% | 0.00e+00 |
| 512 | tiled 32 | 0.555 | 484 | 6 | 19.2% | 0.00e+00 |
| 512 | regblock 4x4 | 0.219 | 1227 | 14 | 48.6% | 0.00e+00 |
| 512 | vec4 + regblock | 0.190 | 1413 | 17 | 55.9% | 0.00e+00 |
| 512 | cuBLAS | 0.106 | 2525 | 30 | 100.0% | 0.00e+00 |
| 1024 | naive | 7.131 | 301 | 2 | 9.1% | 1.62e-05 |
| 1024 | tiled 16 | 4.444 | 483 | 3 | 14.6% | 1.62e-05 |
| 1024 | tiled 32 | 4.116 | 522 | 3 | 15.8% | 1.62e-05 |
| 1024 | regblock 4x4 | 1.482 | 1449 | 8 | 43.8% | 1.62e-05 |
| 1024 | vec4 + regblock | 1.231 | 1744 | 10 | 52.8% | 1.62e-05 |
| 1024 | cuBLAS | 0.650 | 3305 | 19 | 100.0% | 0.00e+00 |
| 2048 | naive | 41.398 | 415 | 1 | 6.2% | 6.10e-05 |
| 2048 | tiled 16 | 26.442 | 650 | 2 | 9.7% | 6.10e-05 |
| 2048 | tiled 32 | 19.518 | 880 | 3 | 13.1% | 6.10e-05 |
| 

## 3. JAX comparison
Colab ships JAX with GPU support. If `jax.devices()` shows only CPU, run `!pip install -U "jax[cuda12]"` and restart.

In [7]:
%%writefile jax_bench.py
"""
The same RMSNorm and SwiGLU in JAX, compiled with jax.jit (XLA fuses them),
benchmarked against our Triton kernels on the same GPU.

Point of the comparison: XLA's automatic fusion vs hand-written Triton kernels.
Run: python jax_bench.py   (after test_and_bench.py, on the same GPU)
"""
import time
import numpy as np
import jax
import jax.numpy as jnp


@jax.jit
def rmsnorm_jax(x, w, eps=1e-6):
    xf = x.astype(jnp.float32)
    rstd = jax.lax.rsqrt(jnp.mean(xf * xf, axis=-1, keepdims=True) + eps)
    return (xf * rstd * w.astype(jnp.float32)).astype(x.dtype)


@jax.jit
def swiglu_jax(g, u):
    gf = g.astype(jnp.float32)
    return (jax.nn.silu(gf) * u.astype(jnp.float32)).astype(g.dtype)


# jax.grad works on our function for free: JAX differentiates the traced math.
rmsnorm_grad = jax.jit(jax.grad(lambda x, w: rmsnorm_jax(x, w).astype(jnp.float32).sum(), argnums=(0, 1)))


def bench_ms(f, *args, iters=50):
    jax.block_until_ready(f(*args))              # first call compiles; keep it out of the timing
    t0 = time.perf_counter()
    for _ in range(iters):
        out = f(*args)
    jax.block_until_ready(out)
    return (time.perf_counter() - t0) / iters * 1e3


def check():
    """JAX matches a NumPy reference."""
    rng = np.random.default_rng(0)
    x = rng.standard_normal((8, 1000)).astype(np.float32)
    w = rng.standard_normal(1000).astype(np.float32)
    ref = x / np.sqrt((x * x).mean(-1, keepdims=True) + 1e-6) * w
    np.testing.assert_allclose(np.asarray(rmsnorm_jax(x, w)), ref, rtol=1e-4, atol=1e-4)
    g, u = x, x[::-1].copy()
    ref = g / (1 + np.exp(-g)) * u
    np.testing.assert_allclose(np.asarray(swiglu_jax(g, u)), ref, rtol=1e-4, atol=1e-4)
    dx, dw = rmsnorm_grad(x, w)
    assert dx.shape == x.shape and dw.shape == w.shape
    print("JAX correctness checks passed on", jax.devices()[0].platform)


if __name__ == "__main__":
    check()
    if jax.devices()[0].platform == "gpu":
        rows = 4096
        print(f"\n| Kernel (JAX jit, fp16) | Hidden size | ms | GB/s |")
        print("| --- | --- | --- | --- |")
        for cols in [1024, 2048, 4096, 8192]:
            key = jax.random.PRNGKey(0)
            x = jax.random.normal(key, (rows, cols), dtype=jnp.float16)
            w = jax.random.normal(key, (cols,), dtype=jnp.float16)
            ms = bench_ms(rmsnorm_jax, x, w)
            print(f"| rmsnorm | {cols} | {ms:.3f} | {2 * x.size * 2 / (ms * 1e-3) / 1e9:.0f} |")
            u = jax.random.normal(jax.random.PRNGKey(1), (rows, cols), dtype=jnp.float16)
            ms = bench_ms(swiglu_jax, x, u)           # separate buffers: passing x twice lets XLA read it once
            print(f"| swiglu | {cols} | {ms:.3f} | {3 * x.size * 2 / (ms * 1e-3) / 1e9:.0f} |")


Writing jax_bench.py


In [8]:
import jax; print(jax.devices())
!python jax_bench.py

[CudaDevice(id=0)]
JAX correctness checks passed on gpu

| Kernel (JAX jit, fp16) | Hidden size | ms | GB/s |
| --- | --- | --- | --- |
| rmsnorm | 1024 | 0.359 | 47 |
| swiglu | 1024 | 0.416 | 61 |
| rmsnorm | 2048 | 0.212 | 159 |
| swiglu | 2048 | 0.202 | 250 |
| rmsnorm | 4096 | 0.404 | 166 |
| swiglu | 4096 | 0.393 | 256 |
| rmsnorm | 8192 | 0.783 | 171 |
| swiglu | 8192 | 0.775 | 260 |


## 4. Explain your results (write these yourself, in README.md)

1. Which Triton kernel beat PyTorch eager by the most, and why?
2. Where did torch.compile or JAX match your kernel? Why does that make sense?
3. What did each GEMM step (tiling, register blocking, float4) gain at N=4096? What % of cuBLAS did you reach?
4. Derive the softmax backward formula dx = y * (dy - sum(dy*y)) on paper.
5. What would you do next to close the gap to cuBLAS?